# 🧪 معمل التجارب: مخزن المتجهات في الذاكرة الحية InMemoryVectorStore (Lecture 34)
أهلاً بك في معملك التطبيقي! 🚀

**فكرة المعمل ببساطة:**
في هذا الدرس نتعرف على أبسط وأخف مخزن متجهات في لانج تشين: **`InMemoryVectorStore`**.
على عكس قواعد البيانات الأخرى، هذا المخزن:
1. **بدون أي تثبيت خارجي (Zero Overhead):** مدمج أصلياً داخل حزمة `langchain-core` مباشرة، ولا يتطلب SQLite أو ملفات ثنائية C++ أو أي خوادم سحابية.
2. **يعمل في الرام 100%:** يخزن المتجهات داخل قاموس بايثون عادي (`dict`)، ويحسب تشابه جيب التمام (Cosine Similarity) باستخدام مكتبة **NumPy**.
3. **مثالي للاختبارات السريعة (Unit Testing):** ممتاز للنماذج الأولية، ولتطبيقات الويب السريعة التي تحتفظ بالسياق المؤقت طوال جلسة المستخدم.

في هذا المعمل ستتعلم عملياً:
- إنشاء المخزن وإضافة المستندات مع البيانات الوصفية واستخراج المعرفات.
- البحث بالتشابه وحساب درجات الـ Cosine Similarity.
- الفلترة بالـ Metadata باستخدام دوال Lambda.
- التحويل لـ Retriever وبناء مسار RAG كامل باستخدام **LCEL** ونموذج **Groq**.

> 💡 **طريقة التدريب في المعمل:**
> اقرأ الشرح المبسط ⬅️ شوف الكود المرجعي الجاهز ⬅️ واكتب الكود وجربه بإيدك في الخلية الفارغة المخصصة لك تحت كل خطوة!

### 1. استيراد المكتبات وتجهيز نموذج التضمين (Embeddings)
**الفكرة ببساطة:**
بنستورد `InMemoryVectorStore` من حزمة `langchain_core.vectorstores` مباشرة، وبنجهز نموذج التضمين المجاني من Hugging Face لتحويل النصوص إلى متجهات عددية.

In [ ]:
# 1. import standard libraries and load environment variables
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# import in-memory vector store and core components
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

# initialize local embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# initialize in-memory vector store instance
vector_store = InMemoryVectorStore(embeddings)

print("✅ InMemoryVectorStore initialized successfully!")
print("✅ Local HuggingFace Embeddings ready.")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب استيراد InMemoryVectorStore وتجهيز التضمين بنفسك هنا):



### 2. تجهيز قائمة المستندات المتنوعة مع البيانات الوصفية (Metadata)
**الفكرة ببساطة:**
بننشئ قائمة من كائنات `Document` تمثل مجالات متنوعة (تغريدات عن الإفطار، أخبار الطقس، ملاحظات برمجية عن RAG)، مع إرفاق بيانات وصفية (`source`, `topic`) لكل مستند.

In [ ]:
# create sample documents representing multiple topics and sources
sample_documents = [
    Document(
        page_content="I had chocolate chips and scrambled eggs for breakfast this morning.",
        metadata={"source": "tweet", "topic": "food"}
    ),
    Document(
        page_content="The weather forecast for tomorrow is cloudy with scattered rain showers.",
        metadata={"source": "news", "topic": "weather"}
    ),
    Document(
        page_content="Building RAG pipelines with LangChain and LCEL allows clean and modular prompt chaining.",
        metadata={"source": "tech_blog", "topic": "ai"}
    ),
    Document(
        page_content="Severe weather warnings were issued for coastal areas due to an approaching tropical storm.",
        metadata={"source": "news", "topic": "weather"}
    ),
    Document(
        page_content="A balanced breakfast rich in protein and fiber provides sustained energy throughout the day.",
        metadata={"source": "health_article", "topic": "food"}
    ),
    Document(
        page_content="Vector databases index dense embeddings to enable sub-millisecond semantic search at scale.",
        metadata={"source": "tech_blog", "topic": "ai"}
    ),
]

print(f"✅ Prepared {len(sample_documents)} sample documents:")
for idx, doc in enumerate(sample_documents, start=1):
    print(f"  [{idx}] (Topic: {doc.metadata['topic']:<8} | Source: {doc.metadata['source']:<14}): {doc.page_content[:60]}...")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب إنشاء قائمة مستندات مع الـ metadata بنفسك هنا):



### 3. إضافة المستندات للمخزن واستعراض المعرفات (`add_documents`)
**الفكرة ببساطة:**
بنضيف المستندات للمخزن بواسطة دالة `add_documents`، والمخزن بيولد تلقائياً معرّفاً فريداً (UUID) لكل مستند مخزن في الذاكرة الحية.

In [ ]:
# add documents to in-memory store
doc_ids = vector_store.add_documents(documents=sample_documents)

print(f"✅ Successfully added {len(doc_ids)} documents to InMemoryVectorStore!")
print("🔑 Generated Document UUIDs:")
for idx, doc_id in enumerate(doc_ids, start=1):
    print(f"  [{idx}] {doc_id}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب إضافة المستندات واستعراض الـ IDs بنفسك هنا):



### 4. البحث بالتشابه الدلالي (Similarity Search)
**الفكرة ببساطة:**
بنسأل سؤال عن حالة الطقس: *"How's the weather forecast?"* ونطلب استرجاع أفضل نتيجتين (`k=2`).
المخزن بيحسب تشابه المتجهات ويستخرج النصوص الأكثر ارتباطاً بالمعنى حتى لو الكلمات مختلفة.

In [ ]:
# query the in-memory vector store
query = "How is the weather forecast looking?"
results = vector_store.similarity_search(query, k=2)

print(f"❓ Query: '{query}'\n")
print(f"🔍 Top {len(results)} Relevant Results:\n")
for idx, doc in enumerate(results, start=1):
    print(f"  [{idx}] Topic: {doc.metadata['topic']} | Source: {doc.metadata['source']}")
    print(f"      Content: {doc.page_content}\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب إجراء بحث دلالي بسؤال من اختيارك بنفسك هنا):



### 5. البحث مع درجات التشابه الدلالي (Cosine Similarity Scores)
**الفكرة ببساطة:**
في `InMemoryVectorStore`، دالة `similarity_search_with_score` بتعيد أزواجاً من `(document, score)`.
الدرجة هنا تمثل **تشابه جيب التمام (Cosine Similarity)**:
الرقم الأقرب إلى `1.0` يعني تطابقاً أعلى، والأقرب إلى `0.0` يعني انخفاض التشابه.

In [ ]:
# similarity search returning (document, cosine_score) pairs
scored_results = vector_store.similarity_search_with_score(query, k=3)

print(f"❓ Query: '{query}'\n")
print("📊 Results with Cosine Similarity Scores (HIGHER = MORE SIMILAR):\n")
for idx, (doc, score) in enumerate(scored_results, start=1):
    print(f"  Rank [{idx}] -> Cosine Similarity: {score:.4f}")
    print(f"    Topic:   {doc.metadata['topic']} | Source: {doc.metadata['source']}")
    print(f"    Content: {doc.page_content}\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب استخراج درجات التشابه وفحصها بنفسك هنا):



### 6. التصفية باستخدام البيانات الوصفية (Metadata Filtering)
**الفكرة ببساطة:**
لو عايزين نبحث فقط في قسم الأغذية والوجبات (`topic == 'food'`)، بنمرر دالة تصفية بسيطة `filter=lambda d: d.metadata.get('topic') == 'food'`، والمخزن هيستبعد تلقائياً أي وثيقة في موضوع آخر.

In [ ]:
# search with lambda metadata filter
food_query = "What did someone eat in the morning?"
food_results = vector_store.similarity_search(
    query=food_query,
    k=2,
    filter=lambda doc: doc.metadata.get("topic") == "food"
)

print(f"🔍 Query: '{food_query}' (Filter: topic == 'food')\n")
print(f"✅ Filtered Results count: {len(food_results)}\n")
for idx, doc in enumerate(food_results, start=1):
    print(f"  [{idx}] Topic: {doc.metadata['topic']} | Source: {doc.metadata['source']}")
    print(f"      Content: {doc.page_content}\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب البحث مع فلترة metadata بموضوع من اختيارك هنا):



### 7. تحويل المخزن إلى مسترجع (Retriever) وتجربة الاستدعاء
**الفكرة ببساطة:**
بنحول المخزن إلى كائن `Retriever` عبر ميثود `.as_retriever()` مع تحديد `k=2`، ونختبر استدعاءه مباشرة بـ `.invoke()` تمهيداً لدمجه في مسار RAG كامل.

In [ ]:
# convert in-memory vector store to retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# test retriever invocation
test_query = "What are the benefits of vector databases in AI?"
retrieved_docs = retriever.invoke(test_query)

print(f"🔍 Retriever invoked for query: '{test_query}'\n")
print(f"Retrieved {len(retrieved_docs)} document chunks:")
for idx, doc in enumerate(retrieved_docs, start=1):
    print(f"  [{idx}] (Source: {doc.metadata['source']}): {doc.page_content}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تحويل المخزن لـ retriever واستدعائه بنفسك هنا):



### 8. بناء مسار RAG كامل باستخدام LCEL ومشغل الربط (`|`) ونموذج Groq
**الفكرة ببساطة:**
بنربط المسترجع مع نموذج الذكاء الاصطناعي فائق السرعة **Groq** وقالب الـ Prompt بمشغل الربط `|`، وبنطرح سؤالاً ونحصل على إجابة ذكية فورية معتمدة على المستندات المحفوظة في الذاكرة الحية!

In [ ]:
# import lcel components and groq chat model
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# helper function to combine retrieved chunks
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# strict context-grounded prompt template
template = """You are a concise and helpful AI assistant.
Answer the question using ONLY the provided context. If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)

# initialize fast Groq LLM
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512
)

# assemble complete LCEL RAG pipeline
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# test query on the in-memory RAG pipeline
rag_query = "What is the weather forecast for tomorrow and what precautions are needed?"
answer = rag_chain.invoke(rag_query)

print(f"❓ Question: {rag_query}\n")
print(f"💬 Answer from InMemory RAG:\n{answer}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب بناء وتشغيل مسار LCEL RAG المتكامل بنفسك هنا):



## 🎯 ملخص المعمل السريع (Lecture 34):
1. **`InMemoryVectorStore`:** مخزن خفيف ومباشر مبني بلغة بايثون الخالصة و NumPy، يأتي مدمجاً في `langchain-core` دون أي مكتبات إضافية.
2. **بدون حفظ دائم (Ephemeral):** يعيش في الذاكرة الحية (RAM) فقط؛ بمجرد إغلاق البرنامج تُمسح البيانات، مما يجعله مثالياً للاختبارات (Unit Tests) والجلسات السريعة.
3. **التصفية المرنة:** يدعم تمرير دوال `filter` مخصصة (كالـ Lambdas) لفحص أي حقل في الـ Metadata بسهولة.
4. **التوافق التام مع LCEL:** يتحول لمسترجع بـ `.as_retriever()` ويتكامل بسلاسة مع سلاسل الـ RAG مثل أي قاعدة بيانات عملاقة.